In [1]:
# Importación de librerías
import os
import time
import pandas as pd
import pyarrow.parquet as pq
import urllib
from sqlalchemy import create_engine
import glob

In [12]:
ruta_archivo = "Datos/Taxi Amarillo/2015/yellow_tripdata_2015-02.parquet"
inicio_total = time.time()

print("1. Leyendo archivo Parquet con PyArrow...")
table = pq.read_table(ruta_archivo)
df_month = table.to_pandas()
print(
    f"-> Leídas {len(df_month):,} filas en {time.time() - inicio_total:.2f}"
    " segundos."
)

# Inspeccionar las columnas originales y las primeras 5 filas
print(f"\nColumnas originales ({len(df_month.columns)}):")
print(list(df_month.columns))

print("\nPrimeras 5 filas del archivo original:")
display(df_month.head(5))

1. Leyendo archivo Parquet con PyArrow...
-> Leídas 12,442,394 filas en 0.74 segundos.

Columnas originales (19):
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Primeras 5 filas del archivo original:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-02-01 00:06:53,2015-02-01 00:09:21,1,0.0,1,N,25,25,3,3.5,0.5,0.5,0.0,0.00,0.3,4.80,None,None
1,1,2015-02-01 00:03:59,2015-02-01 00:11:36,1,5.8,1,N,238,243,2,17.0,0.5,0.5,0.0,0.00,0.3,18.30,None,None
2,1,2015-02-01 00:21:17,2015-02-01 00:36:38,1,7.1,1,N,243,263,1,22.0,0.5,0.5,5.8,0.00,0.3,29.10,None,None
3,1,2015-02-01 00:37:53,2015-02-01 00:57:33,1,11.8,1,N,263,257,1,33.0,0.5,0.5,5.0,5.33,0.3,44.63,None,None
4,1,2015-02-01 00:06:37,2015-02-01 00:15:40,1,1.9,1,N,263,41,1,8.5,0.5,0.5,0.0,0.00,0.3,9.80,None,None


In [2]:
# 1. Crear el motor de SQLAlchemy usando la misma configuración
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=NYC_TLC_DB;"
    "Trusted_Connection=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [3]:
ruta_archivo = "Datos/Taxi Amarillo/2015/yellow_tripdata_2015-01.parquet"
csv_temporal = "temp_yellow_tripdata.csv"

inicio_total = time.time()

print("1. Leyendo archivo Parquet con PyArrow...")
table = pq.read_table(ruta_archivo)
df_month = table.to_pandas()
print(
    f"-> Leídas {len(df_month):,} filas en {time.time() - inicio_total:.2f}"
    " segundos."
)

1. Leyendo archivo Parquet con PyArrow...
-> Leídas 12,741,035 filas en 0.72 segundos.


In [4]:
# Detectar columna de fecha dinámicamente
col_fecha = (
    "tpep_pickup_datetime"
    if "tpep_pickup_datetime" in df_month.columns
    else "lpep_pickup_datetime"
)

# Filtrar fechas
df_month[col_fecha] = pd.to_datetime(df_month[col_fecha], errors="coerce")
lower_bound = pd.Timestamp("2015-01-01 00:00:00")
upper_bound = pd.Timestamp("2017-12-31 23:59:59")

df_filtered = df_month[
    (df_month[col_fecha] >= lower_bound) & (df_month[col_fecha] <= upper_bound)
]
print(f"-> Filas válidas tras el filtro: {len(df_filtered):,}")

-> Filas válidas tras el filtro: 12,741,035


In [5]:
# Mostrar las primeras 5 filas y las columnas actuales
print(f"Columnas actuales ({len(df_filtered.columns)}):")
print(list(df_filtered.columns))
print("\nMuestra de datos:")
display(df_filtered.head(5))

Columnas actuales (19):
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Muestra de datos:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,None,None
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,None,None
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,None,None
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None


In [6]:
if not df_filtered.empty:
  # Normalizar nombres a minúsculas por si acaso
  df_filtered.columns = df_filtered.columns.str.lower()

  # Convertir todo a string y limpiar nulos para el BULK INSERT
  df_filtered = df_filtered.astype(str)
  df_filtered = df_filtered.replace(["None", "nan", "<NA>"], "")

  print("2. Generando CSV temporal...")
  t_csv = time.time()
  df_filtered.to_csv(csv_temporal, index=False, header=False, sep=",")
  print(f"-> CSV temporal creado en {time.time() - t_csv:.2f} segundos.")

  # 3. Ejecutar BULK INSERT masivo en SQL Server
  print("3. Ejecutando BULK INSERT en SQL Server...")
  t_bulk = time.time()
  ruta_csv_sql = os.path.abspath(csv_temporal).replace("\\", "/")

  bulk_query = f"""
    BULK INSERT bronze.yellow_tripdata
    FROM '{ruta_csv_sql}'
    WITH (
        FORMAT = 'CSV',
        DATAFILETYPE = 'char',
        FIELDTERMINATOR = ',',
        ROWTERMINATOR = '\\n',
        BATCHSIZE = 100000,
        TABLOCK
    )
    """

  with engine.begin() as conn:
    conn.exec_driver_sql(bulk_query)

  print(
      f"-> ¡BULK INSERT completado en {time.time() - t_bulk:.2f} segundos!"
  )

  # Limpiar archivo temporal
  if os.path.exists(csv_temporal):
    os.remove(csv_temporal)

  print(
      f"¡Proceso total finalizado con éxito en {time.time() - inicio_total:.2f}"
      " segundos!"
  )
else:
  print("⚠️ El DataFrame quedó vacío tras el filtro de fechas.")

2. Generando CSV temporal...
-> CSV temporal creado en 47.89 segundos.
3. Ejecutando BULK INSERT en SQL Server...
-> ¡BULK INSERT completado en 98.67 segundos!
¡Proceso total finalizado con éxito en 352.78 segundos!


### Validación de todo 2015


In [11]:
# Buscar todos los archivos parquet del 2015
patron_archivos = "Datos/Taxi Amarillo/2015/yellow_tripdata_2015-*.parquet"
archivos = glob.glob(patron_archivos)

print(
    f"Se encontraron {len(archivos)} archivos para revisar en el año 2015.\n"
)

estructura_base = None
todos_iguales = True

for ruta in sorted(archivos):
  # Leer solo los metadatos o las columnas del archivo sin cargar todo el dataframe
  pf = pq.ParquetFile(ruta)
  columnas_actuales = pf.schema.names

  print(
      f"Archivo: {os.path.basename(ruta)} -> {len(columnas_actuales)} columnas"
  )

  if estructura_base is None:
    estructura_base = columnas_actuales
  elif columnas_actuales != estructura_base:
    print(
        f"⚠️ ¡ALERTA! El archivo {os.path.basename(ruta)} tiene columnas"
        " diferentes."
    )
    todos_iguales = False

print("-" * 50)
if todos_iguales:
  print(
      "✅ ¡Excelente noticia! Todos los archivos de 2015 tienen exactamente las"
      " mismas columnas y en el mismo orden."
  )
else:
  print("❌ Se encontraron discrepancias entre las estructuras de los archivos.")

Se encontraron 12 archivos para revisar en el año 2015.

Archivo: yellow_tripdata_2015-01.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-02.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-03.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-04.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-05.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-06.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-07.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-08.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-09.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-10.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-11.parquet -> 19 columnas
Archivo: yellow_tripdata_2015-12.parquet -> 19 columnas
--------------------------------------------------
✅ ¡Excelente noticia! Todos los archivos de 2015 tienen exactamente las mismas columnas y en el mismo orden.


### Inserción de todo 2015


In [13]:
# Buscar todos los archivos del 2015 y excluir el 01
patron_archivos = "Datos/Taxi Amarillo/2015/yellow_tripdata_2015-*.parquet"
archivos = sorted(glob.glob(patron_archivos))

# Filtrar para omitir el mes 01
archivos_a_procesar = [f for f in archivos if not f.endswith("2015-01.parquet")]

print(
    f"Se procesarán {len(archivos_a_procesar)} archivos (desde febrero hasta"
    " diciembre de 2015).\n"
)

Se procesarán 11 archivos (desde febrero hasta diciembre de 2015).



In [14]:
inicio_total_anio = time.time()

for i, ruta_archivo in enumerate(archivos_a_procesar, start=2):
  print(
      f"\n--- Procesando archivo {i}/12: {os.path.basename(ruta_archivo)} ---"
  )
  inicio_mes = time.time()
  csv_temporal = "temp_yellow_tripdata.csv"

  # 1. Lectura con PyArrow
  print("1. Leyendo archivo Parquet...")
  table = pq.read_table(ruta_archivo)
  df_month = table.to_pandas()
  print(f"   -> Leídas {len(df_month):,} filas.")

  # Detección de columna de fecha
  col_fecha = (
      "tpep_pickup_datetime"
      if "tpep_pickup_datetime" in df_month.columns
      else "lpep_pickup_datetime"
  )

  # Filtrar fechas
  df_month[col_fecha] = pd.to_datetime(df_month[col_fecha], errors="coerce")
  lower_bound = pd.Timestamp("2015-01-01 00:00:00")
  upper_bound = pd.Timestamp("2017-12-31 23:59:59")

  df_filtered = df_month[
      (df_month[col_fecha] >= lower_bound)
      & (df_month[col_fecha] <= upper_bound)
  ]
  print(f"   -> Filas válidas tras el filtro: {len(df_filtered):,}")

  if not df_filtered.empty:
    # Normalizar, convertir a string y limpiar nulos
    df_filtered.columns = df_filtered.columns.str.lower()
    df_filtered = df_filtered.astype(str)
    df_filtered = df_filtered.replace(["None", "nan", "<NA>"], "")

    # 2. Generar CSV temporal
    t_csv = time.time()
    df_filtered.to_csv(csv_temporal, index=False, header=False, sep=",")
    print(f"   -> CSV temporal creado en {time.time() - t_csv:.2f}s.")

    # 3. BULK INSERT en SQL Server
    t_bulk = time.time()
    ruta_csv_sql = os.path.abspath(csv_temporal).replace("\\", "/")

    bulk_query = f"""
        BULK INSERT bronze.yellow_tripdata
        FROM '{ruta_csv_sql}'
        WITH (
            FORMAT = 'CSV',
            DATAFILETYPE = 'char',
            FIELDTERMINATOR = ',',
            ROWTERMINATOR = '\\n',
            BATCHSIZE = 100000,
            TABLOCK
        )
        """

    with engine.begin() as conn:
      conn.exec_driver_sql(bulk_query)

    print(f"   -> BULK INSERT completado en {time.time() - t_bulk:.2f}s.")

    # 4. Limpiar archivo temporal
    if os.path.exists(csv_temporal):
      os.remove(csv_temporal)

    print(
        f"   ✨ Mes completado en un total de {time.time() - inicio_mes:.2f}"
        " segundos."
    )
  else:
    print("   ⚠️ El DataFrame quedó vacío tras el filtro de fechas.")

print(f"\n==================================================")
print(
    f"🎉 ¡Año 2015 (meses 02 al 12) cargado con éxito en"
    f" {time.time() - inicio_total_anio:.2f} segundos!"
)
print(f"==================================================")


--- Procesando archivo 2/12: yellow_tripdata_2015-02.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 12,442,394 filas.
   -> Filas válidas tras el filtro: 12,442,394
   -> CSV temporal creado en 42.82s.
   -> BULK INSERT completado en 84.02s.
   ✨ Mes completado en un total de 206.27 segundos.

--- Procesando archivo 3/12: yellow_tripdata_2015-03.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 13,342,951 filas.
   -> Filas válidas tras el filtro: 13,342,951
   -> CSV temporal creado en 59.57s.
   -> BULK INSERT completado en 107.87s.
   ✨ Mes completado en un total de 247.89 segundos.

--- Procesando archivo 4/12: yellow_tripdata_2015-04.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 13,063,758 filas.
   -> Filas válidas tras el filtro: 13,063,758
   -> CSV temporal creado en 51.20s.
   -> BULK INSERT completado en 103.61s.
   ✨ Mes completado en un total de 239.62 segundos.

--- Procesando archivo 5/12: yellow_tripdata_2015-05.parquet ---
1. Leyendo archivo Parq

### Validación de Año 2016

In [16]:
ruta_archivo = "Datos/Taxi Amarillo/2016/yellow_tripdata_2016-01.parquet"
inicio_total = time.time()

print("1. Leyendo archivo Parquet con PyArrow...")
table = pq.read_table(ruta_archivo)
df_month = table.to_pandas()
print(
    f"-> Leídas {len(df_month):,} filas en {time.time() - inicio_total:.2f}"
    " segundos."
)

# Inspeccionar las columnas originales y las primeras 5 filas
print(f"\nColumnas originales ({len(df_month.columns)}):")
print(list(df_month.columns))

print("\nPrimeras 5 filas del archivo original:")
display(df_month.head(5))

1. Leyendo archivo Parquet con PyArrow...
-> Leídas 10,905,067 filas en 0.86 segundos.

Columnas originales (19):
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Primeras 5 filas del archivo original:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2016-01-01 00:12:22,2016-01-01 00:29:14,1,3.2,1,N,48,262,1,14.0,0.5,0.5,3.06,0.0,0.3,18.36,None,None
1,1,2016-01-01 00:41:31,2016-01-01 00:55:10,2,1.0,1,N,162,48,2,9.5,0.5,0.5,0.00,0.0,0.3,10.80,None,None
2,1,2016-01-01 00:53:37,2016-01-01 00:59:57,1,0.9,1,N,246,90,2,6.0,0.5,0.5,0.00,0.0,0.3,7.30,None,None
3,1,2016-01-01 00:13:28,2016-01-01 00:18:07,1,0.8,1,N,170,162,2,5.0,0.5,0.5,0.00,0.0,0.3,6.30,None,None
4,1,2016-01-01 00:33:04,2016-01-01 00:47:14,1,1.8,1,N,161,140,2,11.0,0.5,0.5,0.00,0.0,0.3,12.30,None,None


### Validación de todo el año 2016

In [19]:
# Buscar todos los archivos parquet del 2016
patron_archivos = "Datos/Taxi Amarillo/2016/yellow_tripdata_2016-*.parquet"
archivos = glob.glob(patron_archivos)

print(
    f"Se encontraron {len(archivos)} archivos para revisar en el año 2016.\n"
)

estructura_base = None
todos_iguales = True

for ruta in sorted(archivos):
  # Leer solo los metadatos o las columnas del archivo sin cargar todo el dataframe
  pf = pq.ParquetFile(ruta)
  columnas_actuales = pf.schema.names

  print(
      f"Archivo: {os.path.basename(ruta)} -> {len(columnas_actuales)} columnas"
  )

  if estructura_base is None:
    estructura_base = columnas_actuales
  elif columnas_actuales != estructura_base:
    print(
        f"⚠️ ¡ALERTA! El archivo {os.path.basename(ruta)} tiene columnas"
        " diferentes."
    )
    todos_iguales = False

print("-" * 50)
if todos_iguales:
  print(
      "✅ ¡Excelente noticia! Todos los archivos de 2016 tienen exactamente las"
      " mismas columnas y en el mismo orden."
  )
else:
  print("❌ Se encontraron discrepancias entre las estructuras de los archivos.")

Se encontraron 12 archivos para revisar en el año 2016.

Archivo: yellow_tripdata_2016-01.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-02.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-03.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-04.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-05.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-06.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-07.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-08.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-09.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-10.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-11.parquet -> 19 columnas
Archivo: yellow_tripdata_2016-12.parquet -> 19 columnas
--------------------------------------------------
✅ ¡Excelente noticia! Todos los archivos de 2016 tienen exactamente las mismas columnas y en el mismo orden.


### Validación 2017


In [17]:
ruta_archivo = "Datos/Taxi Amarillo/2017/yellow_tripdata_2017-02.parquet"
inicio_total = time.time()

print("1. Leyendo archivo Parquet con PyArrow...")
table = pq.read_table(ruta_archivo)
df_month = table.to_pandas()
print(
    f"-> Leídas {len(df_month):,} filas en {time.time() - inicio_total:.2f}"
    " segundos."
)

# Inspeccionar las columnas originales y las primeras 5 filas
print(f"\nColumnas originales ({len(df_month.columns)}):")
print(list(df_month.columns))

print("\nPrimeras 5 filas del archivo original:")
display(df_month.head(5))

1. Leyendo archivo Parquet con PyArrow...
-> Leídas 9,169,775 filas en 0.80 segundos.

Columnas originales (19):
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']

Primeras 5 filas del archivo original:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2017-02-01 00:19:20,2017-02-01 00:25:56,1,2.90,1,N,75,162,2,9.5,0.5,0.5,0.00,0.0,0.3,10.80,None,None
1,1,2017-02-01 00:19:55,2017-02-01 00:33:06,1,4.90,1,N,246,166,1,15.0,0.5,0.5,3.25,0.0,0.3,19.55,None,None
2,1,2017-02-01 00:01:15,2017-02-01 00:09:03,2,1.50,1,N,237,170,1,7.5,0.5,0.5,1.50,0.0,0.3,10.30,None,None
3,2,2017-02-01 00:06:36,2017-02-01 00:14:50,5,1.51,1,N,137,236,2,7.5,0.5,0.5,0.00,0.0,0.3,8.80,None,None
4,1,2017-02-01 00:07:53,2017-02-01 00:14:36,1,1.40,1,N,112,112,1,7.0,0.5,0.5,2.45,0.0,0.3,10.75,None,None


### Validación de todo el año 2017

In [20]:
# Buscar todos los archivos parquet del 2017
patron_archivos = "Datos/Taxi Amarillo/2017/yellow_tripdata_2017-*.parquet"
archivos = glob.glob(patron_archivos)

print(
    f"Se encontraron {len(archivos)} archivos para revisar en el año 2017.\n"
)

estructura_base = None
todos_iguales = True

for ruta in sorted(archivos):
  # Leer solo los metadatos o las columnas del archivo sin cargar todo el dataframe
  pf = pq.ParquetFile(ruta)
  columnas_actuales = pf.schema.names

  print(
      f"Archivo: {os.path.basename(ruta)} -> {len(columnas_actuales)} columnas"
  )

  if estructura_base is None:
    estructura_base = columnas_actuales
  elif columnas_actuales != estructura_base:
    print(
        f"⚠️ ¡ALERTA! El archivo {os.path.basename(ruta)} tiene columnas"
        " diferentes."
    )
    todos_iguales = False

print("-" * 50)
if todos_iguales:
  print(
      "✅ ¡Excelente noticia! Todos los archivos de 2017 tienen exactamente las"
      " mismas columnas y en el mismo orden."
  )
else:
  print("❌ Se encontraron discrepancias entre las estructuras de los archivos.")

Se encontraron 12 archivos para revisar en el año 2017.

Archivo: yellow_tripdata_2017-01.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-02.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-03.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-04.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-05.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-06.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-07.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-08.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-09.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-10.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-11.parquet -> 19 columnas
Archivo: yellow_tripdata_2017-12.parquet -> 19 columnas
--------------------------------------------------
✅ ¡Excelente noticia! Todos los archivos de 2017 tienen exactamente las mismas columnas y en el mismo orden.


### Insersición de datos 2016 y 2017

In [21]:
anios_a_procesar = [2016, 2017]

inicio_proceso_total = time.time()

for anio in anios_a_procesar:
  print(f"\n==================================================")
  print(f"🚀 INICIANDO CARGA PARA EL AÑO {anio}")
  print(f"==================================================")

  patron_archivos = f"Datos/Taxi Amarillo/{anio}/yellow_tripdata_{anio}-*.parquet"
  archivos_a_procesar = sorted(glob.glob(patron_archivos))

  print(
      f"Se encontraron {len(archivos_a_procesar)} archivos para el año {anio}.\n"
  )

  inicio_anio = time.time()

  for i, ruta_archivo in enumerate(archivos_a_procesar, start=1):
    print(
        f"\n--- Procesando archivo {i}/12 del año {anio}:"
        f" {os.path.basename(ruta_archivo)} ---"
    )
    inicio_mes = time.time()
    csv_temporal = "temp_yellow_tripdata.csv"

    # 1. Lectura con PyArrow
    print("1. Leyendo archivo Parquet...")
    table = pq.read_table(ruta_archivo)
    df_month = table.to_pandas()
    print(f"   -> Leídas {len(df_month):,} filas.")

    # Detección de columna de fecha
    col_fecha = (
        "tpep_pickup_datetime"
        if "tpep_pickup_datetime" in df_month.columns
        else "lpep_pickup_datetime"
    )

    # Filtrar fechas por seguridad del rango global
    df_month[col_fecha] = pd.to_datetime(df_month[col_fecha], errors="coerce")
    lower_bound = pd.Timestamp("2015-01-01 00:00:00")
    upper_bound = pd.Timestamp("2017-12-31 23:59:59")

    df_filtered = df_month[
        (df_month[col_fecha] >= lower_bound)
        & (df_month[col_fecha] <= upper_bound)
    ]
    print(f"   -> Filas válidas tras el filtro: {len(df_filtered):,}")

    if not df_filtered.empty:
      # Normalizar, convertir a string y limpiar nulos
      df_filtered.columns = df_filtered.columns.str.lower()
      df_filtered = df_filtered.astype(str)
      df_filtered = df_filtered.replace(["None", "nan", "<NA>"], "")

      # 2. Generar CSV temporal
      t_csv = time.time()
      df_filtered.to_csv(csv_temporal, index=False, header=False, sep=",")
      print(f"   -> CSV temporal creado en {time.time() - t_csv:.2f}s.")

      # 3. BULK INSERT en SQL Server
      t_bulk = time.time()
      ruta_csv_sql = os.path.abspath(csv_temporal).replace("\\", "/")

      bulk_query = f"""
            BULK INSERT bronze.yellow_tripdata
            FROM '{ruta_csv_sql}'
            WITH (
                FORMAT = 'CSV',
                DATAFILETYPE = 'char',
                FIELDTERMINATOR = ',',
                ROWTERMINATOR = '\\n',
                BATCHSIZE = 100000,
                TABLOCK
            )
            """

      with engine.begin() as conn:
        conn.exec_driver_sql(bulk_query)

      print(f"   -> BULK INSERT completado en {time.time() - t_bulk:.2f}s.")

      # 4. Limpiar archivo temporal
      if os.path.exists(csv_temporal):
        os.remove(csv_temporal)

      print(
          f"   ✨ Mes completado en un total"
          f" of {time.time() - inicio_mes:.2f} segundos."
      )
    else:
      print("   ⚠️ El DataFrame quedó vacío tras el filtro de fechas.")

  print(
      f"\n🎉 ¡Año {anio} cargado por completo en"
      f" {time.time() - inicio_anio:.2f} segundos!"
  )

print(f"\n==================================================")
print(
    f"🏆 ¡Proceso total finalizado en"
    f" {time.time() - inicio_proceso_total:.2f} segundos!"
)
print(f"==================================================")


🚀 INICIANDO CARGA PARA EL AÑO 2016
Se encontraron 12 archivos para el año 2016.


--- Procesando archivo 1/12 del año 2016: yellow_tripdata_2016-01.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 10,905,067 filas.
   -> Filas válidas tras el filtro: 10,905,067
   -> CSV temporal creado en 31.26s.
   -> BULK INSERT completado en 94.88s.
   ✨ Mes completado en un total of 185.72 segundos.

--- Procesando archivo 2/12 del año 2016: yellow_tripdata_2016-02.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 11,375,412 filas.
   -> Filas válidas tras el filtro: 11,375,412
   -> CSV temporal creado en 30.84s.
   -> BULK INSERT completado en 98.52s.
   ✨ Mes completado en un total of 183.33 segundos.

--- Procesando archivo 3/12 del año 2016: yellow_tripdata_2016-03.parquet ---
1. Leyendo archivo Parquet...
   -> Leídas 12,203,824 filas.
   -> Filas válidas tras el filtro: 12,203,824
   -> CSV temporal creado en 33.05s.
   -> BULK INSERT completado en 102.60s.
   ✨ Mes completado e

In [24]:
import pyarrow.dataset as ds

# Apunta a la carpeta raíz donde se encuentra la carpeta '2015' (y las de los demás años)
dataset_path = "Datos/Taxi Amarillo/" 

# Cargar el dataset apuntando a la raíz; PyArrow detectará automáticamente las subcarpetas o archivos
dataset = ds.dataset(dataset_path, format="parquet", partitioning="hive") # O puedes leer sin partitioning si están organizados por carpetas de años

# 1. Obtener el total general de filas al instante
total_filas = dataset.count_rows()
print(f"Total general de filas en todos los archivos Parquet: {total_filas:,}")

# 2. Si quieres ver el conteo específico por año leyendo la columna de fecha:
table = dataset.to_table(columns=["tpep_pickup_datetime"])
df = table.to_pandas()

# Convertir a fecha y extraer el año
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], errors='coerce')
df['anio'] = df['tpep_pickup_datetime'].dt.year

# Agrupar y mostrar resultados por año
conteo_por_anio = df.groupby('anio').size().reset_index(name='total_filas')
print("\nDesglose de filas por año en los Parquet:")
print(conteo_por_anio)

Total general de filas en todos los archivos Parquet: 390,671,363

Desglose de filas por año en los Parquet:
    anio  total_filas
0   2001            3
1   2002            1
2   2003            5
3   2008           35
4   2009           69
5   2015    146039231
6   2016    131131805
7   2017    113500159
8   2018           53
9   2041            1
10  2053            1
